<a href="https://colab.research.google.com/github/kamalahmadov474/Machine_Learning/blob/main/English_Premier_League_Predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from time import time

# Utility Functions

def train_classifier(clf, X_train, y_train):
    start = time()
    clf.fit(X_train, y_train)
    end = time()
    print("Model trained in {:2f} seconds".format(end-start))

def predict_labels(clf, features, target):
    start = time()
    y_pred = clf.predict(features)
    end = time()
    print("Made Predictions in {:2f} seconds".format(end-start))

    acc = sum(target == y_pred) / float(len(y_pred))
    return f1_score(target, y_pred, average='micro'), acc

def model(clf, X_train, y_train, X_test, y_test):
    train_classifier(clf, X_train, y_train)

    f1, acc = predict_labels(clf, X_train, y_train)
    print("Training Info:")
    print("-" * 20)
    print("F1 Score:{}".format(f1))
    print("Accuracy:{}".format(acc))

    f1, acc = predict_labels(clf, X_test, y_test)
    print("Test Metrics:")
    print("-" * 20)
    print("F1 Score:{}".format(f1))
    print("Accuracy:{}".format(acc))

# Load and preprocess EPL 2019/2020 season data

data_file = 'season-1819_csv.csv'

if not path.exists(data_file):
    raise FileNotFoundError(f"{data_file} not found!")

data = pd.read_csv(data_file)
print(data.head())

# Preprocessing
input_filter = ['home_encoded', 'away_encoded', 'HTHG', 'HTAG', 'HS',
                'AS', 'HST', 'AST', 'HR', 'AR']
output_filter = ['FTR']
cols_to_consider = input_filter + output_filter

encoder = LabelEncoder()
data['home_encoded'] = encoder.fit_transform(data['HomeTeam'])
home_encoded_mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_).tolist()))

encoder = LabelEncoder()
data['away_encoded'] = encoder.fit_transform(data['AwayTeam'])
away_encoded_mapping = dict(zip(encoder.classes_, encoder.transform(encoder.classes_).tolist()))

data = data[cols_to_consider]
print(data[data.isna().any(axis=1)])
data = data.dropna(axis=0)

# Training & Testing

X = data[input_filter]
Y = data['FTR']

X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2)

svc_classifier = SVC(random_state=100, kernel='rbf')
lr_classifier = LogisticRegression(multi_class='ovr', max_iter=500)
nbClassifier = GaussianNB()
rfClassifier = RandomForestClassifier()

print()
print("Logistic Regression one vs All Classifier")
print("-" * 20)
model(lr_classifier, X_train, Y_train, X_test, Y_test)

print()
print("Gaussain Naive Bayes Classifier")
print("-" * 20)
model(nbClassifier, X_train, Y_train, X_test, Y_test)

print()
print("Random Forest Classifier")
print("-" * 20)
model(rfClassifier, X_train, Y_train, X_test, Y_test)

# Predicting full season matches to guess winner
print("\nPredicting full season match results with Random Forest...")
full_preds = rfClassifier.predict(X)

data['PredictedFTR'] = full_preds

# Calculate points
points = {}
for idx, row in data.iterrows():
    home = row['home_encoded']
    away = row['away_encoded']
    result = row['PredictedFTR']

    home_team = list(home_encoded_mapping.keys())[list(home_encoded_mapping.values()).index(home)]
    away_team = list(away_encoded_mapping.keys())[list(away_encoded_mapping.values()).index(away)]

    if home_team not in points:
        points[home_team] = 0
    if away_team not in points:
        points[away_team] = 0

    if result == 'H':
        points[home_team] += 3
    elif result == 'A':
        points[away_team] += 3
    elif result == 'D':
        points[home_team] += 1
        points[away_team] += 1

# Display winner
sorted_table = sorted(points.items(), key=lambda x: x[1], reverse=True)
print("\n=== Predicted EPL 2019/2020 Final Standings ===")
for i, (team, pts) in enumerate(sorted_table):
    print(f"{i+1}. {team} - {pts} pts")

print(f"\n🏆 Predicted Winner: {sorted_table[0][0]}")


  Div        Date      HomeTeam        AwayTeam  FTHG  FTAG FTR  HTHG  HTAG  \
0  E0  10/08/2018    Man United       Leicester     2     1   H     1     0   
1  E0  11/08/2018   Bournemouth         Cardiff     2     0   H     1     0   
2  E0  11/08/2018        Fulham  Crystal Palace     0     2   A     0     1   
3  E0  11/08/2018  Huddersfield         Chelsea     0     3   A     0     2   
4  E0  11/08/2018     Newcastle       Tottenham     1     2   A     1     2   

  HTR  ... BbAv<2.5  BbAH  BbAHh  BbMxAHH  BbAvAHH  BbMxAHA  BbAvAHA  PSCH  \
0   H  ...     1.79    17  -0.75     1.75     1.70     2.29     2.21  1.55   
1   H  ...     1.83    20  -0.75     2.20     2.13     1.80     1.75  1.88   
2   A  ...     1.87    22  -0.25     2.18     2.11     1.81     1.77  2.62   
3   A  ...     1.84    23   1.00     1.84     1.80     2.13     2.06  7.24   
4   A  ...     1.81    20   0.25     2.20     2.12     1.80     1.76  4.74   

   PSCD  PSCA  
0  4.07  7.69  
1  3.61  4.70  
2  3.38 

/usr/local/lib/python3.11/dist-packages/sklearn/linear_model/_logistic.py:1256: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. Use OneVsRestClassifier(LogisticRegression(..)) instead. Leave it to its default value to avoid this warning.
  warnings.warn(


Training Info:
--------------------
F1 Score:0.7006578947368421
Accuracy:0.7006578947368421
Made Predictions in 0.016039 seconds
Test Metrics:
--------------------
F1 Score:0.6842105263157895
Accuracy:0.6842105263157895

Gaussain Naive Bayes Classifier
--------------------
Model trained in 0.021320 seconds
Made Predictions in 0.005728 seconds
Training Info:
--------------------
F1 Score:0.6546052631578947
Accuracy:0.6546052631578947
Made Predictions in 0.004054 seconds
Test Metrics:
--------------------
F1 Score:0.6973684210526315
Accuracy:0.6973684210526315

Random Forest Classifier
--------------------
Model trained in 0.440504 seconds
Made Predictions in 0.018221 seconds
Training Info:
--------------------
F1 Score:1.0
Accuracy:1.0
Made Predictions in 0.024213 seconds
Test Metrics:
--------------------
F1 Score:0.6710526315789473
Accuracy:0.6710526315789473

Predicting full season match results with Random Forest...

=== Predicted EPL 2019/2020 Final Standings ===
1. Man City - 96 p